In [1]:
import pandas as pd
import urllib.request
import zipfile
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Скачиваем
url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
urllib.request.urlretrieve(url, "ml-100k.zip")

with zipfile.ZipFile("ml-100k.zip", "r") as zip_ref:
    zip_ref.extractall(".")

users = pd.read_csv("ml-100k/u.user", sep="|",
                     names=["user_id", "age", "gender", "occupation", "zip"])
items = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
                     names=["item_id", "title", "release_date", "video_release_date",
                            "IMDb_URL", "unknown", "Action", "Adventure", "Animation",
                            "Children", "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
                            "Film_Noir", "Horror", "Musical", "Mystery", "Romance", "Sci_Fi",
                            "Thriller", "War", "Western"])
ratings = pd.read_csv("ml-100k/u.data", sep="\t",
                      names=["user_id", "item_id", "rating", "timestamp"])

print(f"Users: {users.shape}")
print(f"Items: {items.shape}")
print(f"Ratings: {ratings.shape}")

Users: (943, 5)
Items: (1682, 24)
Ratings: (100000, 4)


In [3]:
users.head()

,user_id,age,gender,occupation,zip
0,1,24,M,technician,85711
1,2,53,F,other,94043
2,3,23,M,writer,32067
3,4,24,M,technician,43537
4,5,33,F,other,15213


In [5]:
items.head()

,item_id,title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children,...,Fantasy,Film_Noir,Horror,Musical,Mystery,Romance,Sci_Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [6]:
ratings.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")

Используется устройство: cuda


In [8]:
users = pd.read_csv("ml-100k/u.user", sep="|",
                     names=["user_id", "age", "gender", "occupation", "zip"])

items = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
                     names=["item_id", "title", "release_date", "video_release_date",
                            "IMDb_URL", "unknown", "Action", "Adventure", "Animation",
                            "Children", "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
                            "Film_Noir", "Horror", "Musical", "Mystery", "Romance", "Sci_Fi",
                            "Thriller", "War", "Western"])

ratings = pd.read_csv("ml-100k/u.data", sep="\t",
                      names=["user_id", "item_id", "rating", "timestamp"])

In [9]:
print(f"Пользователей: {users.shape[0]}")
print(f"Фильмов: {items.shape[0]}")
print(f"Оценок: {ratings.shape[0]}")
print(f"Диапазон оценок: {ratings['rating'].min()} - {ratings['rating'].max()}")

Пользователей: 943
Фильмов: 1682
Оценок: 100000
Диапазон оценок: 1 - 5


In [23]:
users['age_group'] = pd.cut(users['age'],
                             bins=[0, 18, 25, 35, 45, 55, 65, 100],
                             labels=[0, 1, 2, 3, 4, 5, 6])
users['age_group'] = users['age_group'].astype(int)

# Возрастные категории (бинарные)
users['is_young'] = (users['age'] < 30).astype(int)
users['is_middle'] = ((users['age'] >= 30) & (users['age'] < 50)).astype(int)
users['is_senior'] = (users['age'] >= 50).astype(int)

# Пол в числовом виде
users['gender_encoded'] = (users['gender'] == 'M').astype(int)

# Профессии с группировкой
occupation_groups = {
    'administrator': 0, 'artist': 0, 'doctor': 0, 'educator': 0, 'engineer': 1,
    'entertainment': 2, 'executive': 1, 'healthcare': 0, 'homemaker': 2,
    'lawyer': 1, 'librarian': 0, 'marketing': 1, 'none': 2, 'other': 2,
    'programmer': 1, 'retired': 3, 'salesman': 1, 'scientist': 1,
    'student': 4, 'technician': 1, 'writer': 0
}
users['occ_group'] = users['occupation'].map(occupation_groups).fillna(2)

# Жанры
genre_cols = ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime',
              'Documentary', 'Drama', 'Fantasy', 'Film_Noir', 'Horror', 'Musical',
              'Mystery', 'Romance', 'Sci_Fi', 'Thriller', 'War', 'Western']

# Количество жанров
items['genre_count'] = items[genre_cols].sum(axis=1)

# Год выпуска
def extract_year(date_str):
    if pd.isna(date_str):
        return 1995
    try:
        return int(str(date_str).split('-')[-1])
    except:
        return 1995

items['year'] = items['release_date'].apply(extract_year)

# Современность
current_year = 1998
items['years_since_release'] = current_year - items['year']
items['is_recent'] = (items['years_since_release'] <= 5).astype(int)

# Название фильма
items['title_length'] = items['title'].str.len()
items['title_words'] = items['title'].str.split().str.len()
items['has_year_in_title'] = items['title'].str.contains(r'\(\d{4}\)').astype(int)

# Популярность фильма
item_rating_count = ratings.groupby('item_id')['rating'].count()
items['rating_count'] = items['item_id'].map(item_rating_count).fillna(0)
items['rating_count_log'] = np.log1p(items['rating_count'])

# Средний рейтинг
item_avg_rating = ratings.groupby('item_id')['rating'].mean()
items['avg_rating'] = items['item_id'].map(item_avg_rating).fillna(3.0)

# Стандартное отклонение
item_rating_std = ratings.groupby('item_id')['rating'].std()
items['rating_std'] = items['item_id'].map(item_rating_std).fillna(0)

# Взвешенный рейтинг
global_avg = ratings['rating'].mean()
items['weighted_rating'] = (items['rating_count'] * items['avg_rating'] + 10 * global_avg) / (items['rating_count'] + 10)

# Средний рейтинг пользователя
user_avg_rating = ratings.groupby('user_id')['rating'].mean()
users['user_avg_rating'] = users['user_id'].map(user_avg_rating).fillna(3.0)

# Количество оценок пользователя
user_rating_count = ratings.groupby('user_id')['rating'].count()
users['user_rating_count'] = users['user_id'].map(user_rating_count).fillna(0)
users['user_rating_count_log'] = np.log1p(users['user_rating_count'])

# Жанровые предпочтения пользователя
user_genre_prefs = ratings.merge(items, on='item_id').groupby('user_id')[genre_cols].mean()
users = users.merge(user_genre_prefs, on='user_id', how='left').fillna(0)

# Переименовываем колонки жанров
for genre in genre_cols:
    users = users.rename(columns={genre: f'pref_{genre}'})

# Пользовательские признаки для модели
user_features = [
    'age', 'age_group', 'is_young', 'is_middle', 'is_senior',
    'gender_encoded', 'occ_group', 'user_avg_rating', 'user_rating_count_log',
    'pref_Action', 'pref_Adventure', 'pref_Animation', 'pref_Children',
    'pref_Comedy', 'pref_Crime', 'pref_Documentary', 'pref_Drama',
    'pref_Fantasy', 'pref_Film_Noir', 'pref_Horror', 'pref_Musical',
    'pref_Mystery', 'pref_Romance', 'pref_Sci_Fi', 'pref_Thriller',
    'pref_War', 'pref_Western'
]

# Товарные признаки для модели
item_features = [
    'genre_count', 'year', 'years_since_release', 'is_recent',
    'title_length', 'title_words', 'has_year_in_title',
    'rating_count_log', 'avg_rating', 'rating_std', 'weighted_rating',
    'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime',
    'Documentary', 'Drama', 'Fantasy', 'Film_Noir', 'Horror', 'Musical',
    'Mystery', 'Romance', 'Sci_Fi', 'Thriller', 'War', 'Western'
]

In [30]:
# Позитивные примеры (рейтинг = 5)
pos_data = ratings[ratings['rating'] == 5].copy()
pos_data = pos_data.merge(users, on='user_id').merge(items, on='item_id')
pos_data['label'] = 1
print(f"Позитивных примеров (рейтинг = 5): {len(pos_data)}")

# Негативные примеры (рейтинг <= 2)
neg_data = ratings[ratings['rating'] <= 2].copy()
neg_data = neg_data.merge(users, on='user_id').merge(items, on='item_id')
neg_data['label'] = 0
print(f"Негативных примеров (рейтинг <= 2): {len(neg_data)}")

# Уравниваем классы
if len(neg_data) > len(pos_data):
    neg_data = neg_data.sample(n=len(pos_data), random_state=42)
else:
    pos_data = pos_data.sample(n=len(neg_data), random_state=42)

# Объединяем
full_data = pd.concat([pos_data, neg_data], ignore_index=True)
print(f"Всего примеров: {len(full_data)}")
print(f"Баланс классов: 1={full_data['label'].sum():.0f}, 0={len(full_data)-full_data['label'].sum():.0f}")


Позитивных примеров (рейтинг = 5): 21201
Негативных примеров (рейтинг <= 2): 17480
Всего примеров: 34960
Баланс классов: 1=17480, 0=17480


In [143]:
print(f"\nПризнаки пользователя {user_features}")
print(f"Признаки фильма {item_features}")


Признаки пользователя ['age', 'age_group', 'is_young', 'is_middle', 'is_senior', 'gender_encoded', 'occ_group', 'user_avg_rating', 'user_rating_count_log', 'pref_Action', 'pref_Adventure', 'pref_Animation', 'pref_Children', 'pref_Comedy', 'pref_Crime', 'pref_Documentary', 'pref_Drama', 'pref_Fantasy', 'pref_Film_Noir', 'pref_Horror', 'pref_Musical', 'pref_Mystery', 'pref_Romance', 'pref_Sci_Fi', 'pref_Thriller', 'pref_War', 'pref_Western']
Признаки фильма ['genre_count', 'year', 'years_since_release', 'is_recent', 'title_length', 'title_words', 'has_year_in_title', 'rating_count_log', 'avg_rating', 'rating_std', 'weighted_rating', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film_Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci_Fi', 'Thriller', 'War', 'Western']


In [69]:
X_user = full_data[user_features].values.astype(np.float32)
X_item = full_data[item_features].values.astype(np.float32)
y = full_data['label'].values

print(f"X_user shape: {X_user.shape}")
print(f"X_item shape: {X_item.shape}")

X_user shape: (34960, 45)
X_item shape: (34960, 29)


In [73]:
X_train_user, X_test_user, X_train_item, X_test_item, y_train, y_test = train_test_split(
    X_user, X_item, y, test_size=0.2, random_state=50, stratify=y
)

print(f"Train size: {len(X_train_user)}, Test size: {len(X_test_user)}")

Train size: 27968, Test size: 6992


In [90]:
class RecDataset(Dataset):
    def __init__(self, X_user, X_item, y):
        self.X_user = torch.FloatTensor(X_user)
        self.X_item = torch.FloatTensor(X_item)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_user[idx], self.X_item[idx], self.y[idx]

train_dataset = RecDataset(X_train_user, X_train_item, y_train)
test_dataset = RecDataset(X_test_user, X_test_item, y_test)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

class UserTower(nn.Module):
    def __init__(self, input_dim, emb_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.Sigmoid(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.Sigmoid(),
            nn.Dropout(0.3),
            nn.Linear(256, emb_dim)
        )

    def forward(self, x):
        return self.net(x)

class ItemTower(nn.Module):
    def __init__(self, input_dim, emb_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.Sigmoid(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.Sigmoid(),
            nn.Dropout(0.3),
            nn.Linear(256, emb_dim)
)

    def forward(self, x):
        return self.net(x)

class TwoTower(nn.Module):
    def __init__(self, user_dim, item_dim, emb_dim=512):
        super().__init__()
        self.user_tower = UserTower(user_dim, emb_dim)
        self.item_tower = ItemTower(item_dim, emb_dim)

    def forward(self, user_features, item_features):
        user_emb = self.user_tower(user_features)
        item_emb = self.item_tower(item_features)
        # Нормализуем эмбеддинги для косинусной близости
        user_emb = nn.functional.normalize(user_emb, p=2, dim=1)
        item_emb = nn.functional.normalize(item_emb, p=2, dim=1)
        return (user_emb * item_emb).sum(dim=1)

In [91]:
user_dim = X_user.shape[1]
item_dim = X_item.shape[1]

model = TwoTower(user_dim, item_dim, emb_dim=512).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00002)

epochs = 100
best_auc = 0
best_acc = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for user_batch, item_batch, y_batch in train_loader:
        user_batch = user_batch.to(device)
        item_batch = item_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(user_batch, item_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for user_batch, item_batch, y_batch in test_loader:
            user_batch = user_batch.to(device)
            item_batch = item_batch.to(device)
            logits = model(user_batch, item_batch)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.extend(probs)
            all_labels.extend(y_batch.numpy())

    AUC = roc_auc_score(all_labels, all_preds)
    Accuracy = accuracy_score(all_labels, [1 if p > 0.5 else 0 for p in all_preds])

    if AUC > best_auc:
        best_auc = AUC
        torch.save(model.state_dict(), 'best_two_tower_movielens.pth')

    if Accuracy > best_acc:
        best_acc = Accuracy

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}, Loss: {total_loss/len(train_loader):.4f}, AUC: {AUC:.4f}, Acсuracy: {Accuracy:.4f}")

print(f"\nЛучший AUC: {best_auc:.4f}, Лучший Accuracy: {best_acc:.4f}")

Epoch  5, Loss: 0.6489, AUC: 0.7418, Acсuracy: 0.6715
Epoch 10, Loss: 0.6136, AUC: 0.8108, Acсuracy: 0.6642
Epoch 15, Loss: 0.5791, AUC: 0.8490, Acсuracy: 0.7377
Epoch 20, Loss: 0.5534, AUC: 0.8726, Acсuracy: 0.7404
Epoch 25, Loss: 0.5378, AUC: 0.8787, Acсuracy: 0.7785
Epoch 30, Loss: 0.5298, AUC: 0.8832, Acсuracy: 0.7803
Epoch 35, Loss: 0.5240, AUC: 0.8875, Acсuracy: 0.8069
Epoch 40, Loss: 0.5204, AUC: 0.8869, Acсuracy: 0.8012
Epoch 45, Loss: 0.5160, AUC: 0.8911, Acсuracy: 0.8205
Epoch 50, Loss: 0.5149, AUC: 0.8921, Acсuracy: 0.8234
Epoch 55, Loss: 0.5122, AUC: 0.8915, Acсuracy: 0.8151
Epoch 60, Loss: 0.5117, AUC: 0.8937, Acсuracy: 0.8258
Epoch 65, Loss: 0.5104, AUC: 0.8939, Acсuracy: 0.8207
Epoch 70, Loss: 0.5092, AUC: 0.8943, Acсuracy: 0.8231
Epoch 75, Loss: 0.5080, AUC: 0.8912, Acсuracy: 0.8049
Epoch 80, Loss: 0.5070, AUC: 0.8945, Acсuracy: 0.8275
Epoch 85, Loss: 0.5052, AUC: 0.8949, Acсuracy: 0.8242
Epoch 90, Loss: 0.5058, AUC: 0.8928, Acсuracy: 0.8169
Epoch 95, Loss: 0.5049, AUC:

In [142]:
def recommend_random_from_test(top_n=10):
    model.eval()

    # Выбираем случайный индекс из тестовой выборки
    random_idx = np.random.randint(0, len(X_test_user))
    test_user_vec = X_test_user[random_idx]

    print(f"Индекс: {random_idx}")
    print(f"Пример признаков (Age, AgeGroup): {test_user_vec[0]:.1f}, {test_user_vec[1]:.1f}")

    # Переводим в тензор и нормализуем
    with torch.no_grad():
        user_tensor = torch.FloatTensor(test_user_vec.reshape(1, -1)).to(device)
        user_emb = model.user_tower(user_tensor)
        user_emb = nn.functional.normalize(user_emb, p=2, dim=1)

    # Готовим фильмы
    item_matrix = items[item_features].values.astype(np.float32)
    item_tensor = torch.FloatTensor(item_matrix).to(device)

    with torch.no_grad():
        all_item_embs = model.item_tower(item_tensor)
        all_item_embs = nn.functional.normalize(all_item_embs, p=2, dim=1)

    # Считаем скоры
    scores = torch.matmul(user_emb, all_item_embs.T).cpu().numpy().flatten()

    # Создаем список индексов фильмов, отсортированный по score
    top_indices = np.argsort(scores)[::-1][:top_n]

    print(f"\nТоп-10 рекомендаций (для пользователя из теста):")
    for rank, idx in enumerate(top_indices, 1):
        item_id = items['item_id'].iloc[idx]
        title = items['title'].iloc[idx]
        score = scores[idx]
        print(f"  {rank}. {title} (Score: {score:.4f})")
recommend_random_from_test(top_n=10)

Индекс: 3671
Пример признаков (Age, AgeGroup): 24.0, 1.0

Топ-10 рекомендаций (для пользователя из теста):
  1. Casablanca (1942) (Score: 0.8071)
  2. Rear Window (1954) (Score: 0.8067)
  3. 12 Angry Men (1957) (Score: 0.8027)
  4. To Kill a Mockingbird (1962) (Score: 0.8002)
  5. Schindler's List (1993) (Score: 0.7982)
  6. Star Wars (1977) (Score: 0.7953)
  7. Godfather, The (1972) (Score: 0.7940)
  8. Third Man, The (1949) (Score: 0.7928)
  9. North by Northwest (1959) (Score: 0.7921)
  10. One Flew Over the Cuckoo's Nest (1975) (Score: 0.7901)
